In [2]:
"""
Step 0b — Segment-Based Feature Extraction (5s windows, 25% overlap)
------------------------------------------------------------------------
pip install praat-parselmouth librosa scipy pandas numpy webrtcvad-wheels joblib

Same 135 features as the full-recording extractor, but computed per
5-second segment (3.75s hop, i.e. 25% overlap — matches SpeechCARE's
segmentation on the same PREPARE dataset) instead of once per whole
recording. Each row is one segment, tagged with its parent uid so
segments can be grouped back together later (both for training with
GroupKFold and for aggregating predictions back to person-level).
"""

import os
import warnings
from itertools import groupby
import numpy as np
import pandas as pd
import librosa
import parselmouth
from parselmouth.praat import call
from scipy import stats
import webrtcvad
from pathlib import Path
from joblib import Parallel, delayed

warnings.filterwarnings("ignore", message="Precision loss occurred in moment calculation")

# ── Paths ───────────────────────────────────────────────
METADATA      = r"C:\Users\arday\Downloads\metadata.csv"
TRAIN_FOLDER  = r"C:\Users\arday\Downloads\train_audios"
TEST_FOLDER   = r"C:\Users\arday\Downloads\test_audios"
OUTPUT_DIR    = r"C:\Users\arday\Desktop\full_features_segments"
SEGMENT_LEN     = 5.0    # seconds
HOP_LEN         = 3.75   # seconds (25% overlap)
MIN_SEGMENT_LEN = 3.0    # discard trailing segments shorter than this
# ────────────────────────────────────────────────────────

AUDIO_EXTS = ('.wav', '.mp3', '.flac', '.m4a')


def stats6(arr):
    arr = np.asarray(arr, dtype=float)
    arr = arr[~np.isnan(arr)]
    if len(arr) < 2:
        return [np.nan] * 6
    return [np.mean(arr), np.std(arr), np.min(arr), np.max(arr),
            float(stats.kurtosis(arr)), float(stats.skew(arr))]


# ── Prosody: F0, Intensity, Energy ──────────────────────

def f0_features(pitch):
    f0 = pitch.selected_array['frequency']
    f0 = f0[f0 > 0]
    m, s, mn, mx, ku, sk = stats6(f0)
    return {'f0_mean': m, 'f0_std': s, 'f0_min': mn, 'f0_max': mx,
            'f0_kurtosis': ku, 'f0_skewness': sk,
            'f0_std_norm': s / m if m and m > 0 else np.nan}


def intensity_features(intensity):
    vals = intensity.values.flatten()
    m, s, mn, mx, ku, sk = stats6(vals)
    return {'intensity_mean': m, 'intensity_std': s, 'intensity_min': mn,
            'intensity_max': mx, 'intensity_kurtosis': ku, 'intensity_skewness': sk}


def energy_features(y):
    rms = librosa.feature.rms(y=y, frame_length=512, hop_length=256)[0]
    m, s, mn, mx, ku, sk = stats6(rms)
    return {'energy_mean': m, 'energy_std': s, 'energy_min': mn,
            'energy_max': mx, 'energy_kurtosis': ku, 'energy_skewness': sk}


# ── Pause & Fluency (WebRTC VAD) ─────────────────────────

def pause_features(y, sr=16000):
    total_dur = len(y) / sr
    pcm = (np.clip(y, -1.0, 1.0) * 32767).astype(np.int16)

    vad = webrtcvad.Vad(2)
    frame_ms, frame_len = 20, int(sr * 20 / 1000)

    labels = []
    for i in range(0, len(pcm) - frame_len, frame_len):
        try:
            labels.append(vad.is_speech(pcm[i:i+frame_len].tobytes(), sr))
        except Exception:
            labels.append(True)

    MIN_SILENCE, MIN_SPEECH = 5, 2
    runs = [(k, len(list(g))) for k, g in groupby(labels)]
    merged = []
    for is_speech, count in runs:
        if not is_speech and count < MIN_SILENCE:
            merged.append((True, count))
        elif is_speech and count < MIN_SPEECH:
            merged.append((False, count))
        else:
            merged.append((is_speech, count))

    final = []
    for is_speech, count in merged:
        if final and final[-1][0] == is_speech:
            final[-1] = (is_speech, final[-1][1] + count)
        else:
            final.append((is_speech, count))

    pauses, speech_segs = [], []
    for is_speech, count in final:
        dur = count * frame_ms / 1000
        (speech_segs if is_speech else pauses).append(dur)

    p   = np.array(pauses)      if pauses      else np.array([0.0])
    s   = np.array(speech_segs) if speech_segs else np.array([total_dur])
    sp  = s.sum()
    ipi = s[1:-1] if len(s) > 2 else np.array([0.0])
    n_p = len(pauses)

    return {
        'mean_pause_duration':          float(np.mean(p)) if n_p else 0.0,
        'total_speech_time':            float(sp),
        'total_pause_time':             float(p.sum()) if n_p else 0.0,
        'pause_percentage':             float(p.sum()/total_dur*100) if n_p else 0.0,
        'silence_to_speech_ratio':      float(p.sum()/sp) if n_p and sp>0 else 0.0,
        'pause_variability':            float(np.std(p)) if n_p>1 else 0.0,
        'number_of_pauses':             n_p,
        'number_of_speech_segments':    len(speech_segs),
        'max_pause_duration':           float(np.max(p)) if n_p else 0.0,
        'min_pause_duration':           float(np.min(p)) if n_p else 0.0,
        'pause_rate':                   n_p/total_dur,
        'mean_speech_segment_duration': float(np.mean(s)),
        'std_speech_segment_duration':  float(np.std(s)) if len(s)>1 else 0.0,
        'mean_inter_pause_interval':    float(np.mean(ipi)),
        'std_inter_pause_interval':     float(np.std(ipi)) if len(ipi)>1 else 0.0,
        'long_pause_count':             int(np.sum(p > 0.5)),
        'short_pause_count':            int(np.sum((p>0)&(p<=0.5))),
        'pause_duration_skewness':      float(stats.skew(p)) if len(p)>2 else 0.0,
    }


# ── Articulation & Resonance ──────────────────────────────

def articulation_features(snd, pitch, intensity):
    total_dur = snd.get_total_duration()
    f0 = pitch.selected_array['frequency']
    voiced_dur = np.sum(f0 > 0) * pitch.time_step
    n_syl = max(voiced_dur / 0.15, 1)

    tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
    n = call(tg, "Get number of intervals", 1)
    speech_time = sum(
        call(tg, "Get end time of interval", 1, i) - call(tg, "Get start time of interval", 1, i)
        for i in range(1, n + 1)
        if call(tg, "Get label of interval", 1, i) == "speech"
    )
    return {
        'speech_rate':           n_syl / total_dur,
        'articulation_rate':     n_syl / speech_time if speech_time > 0 else np.nan,
        'avg_syllable_duration': speech_time / n_syl if n_syl > 0 else np.nan,
    }


def formant_features(snd):
    fmt = snd.to_formant_burg(time_step=0.02, max_number_of_formants=5, maximum_formant=5500,
                               window_length=0.025, pre_emphasis_from=50)
    t = fmt.xs()
    f1 = np.array([call(fmt, "Get value at time", 1, ti, 'Hertz', 'Linear') for ti in t])
    f2 = np.array([call(fmt, "Get value at time", 2, ti, 'Hertz', 'Linear') for ti in t])
    f1, f2 = f1[~np.isnan(f1)], f2[~np.isnan(f2)]
    disp = np.mean(f2) - np.mean(f1) if len(f1) and len(f2) else np.nan
    vtl = 34300 / (2 * disp) if disp and disp > 0 else np.nan
    return {'f1_std': np.std(f1) if len(f1) > 1 else np.nan,
            'f2_std': np.std(f2) if len(f2) > 1 else np.nan,
            'vocal_tract_length': vtl}


# ── Voice Quality: Jitter, Shimmer, HNR ──────────────────

def voice_quality_features(snd, window=1.0, hop=0.5):
    total_dur = snd.get_total_duration()
    jl_vals, jd_vals, sl_vals, sa_vals = [], [], [], []

    t = 0.0
    while t + window <= total_dur:
        try:
            part = snd.extract_part(t, t + window, preserve_times=False)
            pp = call(part, "To PointProcess (periodic, cc)", 75, 600)
            if call(pp, "Get number of points") >= 4:
                jl_vals.append(call(pp, "Get jitter (local)", 0, 0, 0.0001, 0.02, 1.3))
                jd_vals.append(call(pp, "Get jitter (ddp)", 0, 0, 0.0001, 0.02, 1.3))
                sl_vals.append(call([part, pp], "Get shimmer (local)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
                sa_vals.append(call([part, pp], "Get shimmer (apq3)", 0, 0, 0.0001, 0.02, 1.3, 1.6))
        except Exception:
            pass
        t += hop

    out = {}
    for name, vals in [('jitter_local', jl_vals), ('jitter_ddp', jd_vals),
                        ('shimmer_local', sl_vals), ('shimmer_apq3', sa_vals)]:
        m, s, mn, mx, ku, sk = stats6(vals)
        out[f'{name}_mean'] = m;  out[f'{name}_std'] = s
        out[f'{name}_min']  = mn; out[f'{name}_max'] = mx
        out[f'{name}_kurtosis'] = ku; out[f'{name}_skewness'] = sk

    harmonicity = snd.to_harmonicity_cc(time_step=0.01, minimum_pitch=75)
    hnr_vals = harmonicity.values.flatten()
    hnr_vals = hnr_vals[hnr_vals > -200]
    m, s, mn, mx, ku, sk = stats6(hnr_vals)
    out['hnr_mean'] = m; out['hnr_std'] = s; out['hnr_min'] = mn
    out['hnr_max'] = mx; out['hnr_kurtosis'] = ku; out['hnr_skewness'] = sk
    return out


# ── Spectral: MFCC / Delta-MFCC (librosa) ────────────────

def mfcc_features(y, sr=16000, n_mfcc=13):
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=n_mfcc)
    delta = librosa.feature.delta(mfcc)
    out = {}
    for i in range(n_mfcc):
        out[f'mfcc_{i+1}_mean'] = float(np.mean(mfcc[i]))
        out[f'mfcc_{i+1}_std']  = float(np.std(mfcc[i]))
    for i in range(n_mfcc):
        out[f'delta_mfcc_{i+1}_mean'] = float(np.mean(delta[i]))
        out[f'delta_mfcc_{i+1}_std']  = float(np.std(delta[i]))
    return out


# ── Spectral Shape: Alpha Ratio, Hammarberg, Flux, Slope ─

def spectral_shape_features(y, sr=16000, n_fft=1024, hop_length=256):
    S = np.abs(librosa.stft(y, n_fft=n_fft, hop_length=hop_length))
    freqs = librosa.fft_frequencies(sr=sr, n_fft=n_fft)
    power = S ** 2

    low, high = (freqs >= 50) & (freqs < 1000), (freqs >= 1000) & (freqs <= 5000)
    e_low, e_high = power[low].sum(axis=0), power[high].sum(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        alpha_ratio = 20 * np.log10(np.where(e_high > 0, e_low / np.where(e_high > 0, e_high, 1), np.nan))

    h1, h2 = (freqs < 2000), (freqs >= 2000) & (freqs <= 5000)
    peak1, peak2 = power[h1].max(axis=0), power[h2].max(axis=0)
    with np.errstate(divide='ignore', invalid='ignore'):
        hammarberg = 20 * np.log10(np.where(peak2 > 0, peak1 / np.where(peak2 > 0, peak2, 1), np.nan))

    S_norm = S / (S.sum(axis=0, keepdims=True) + 1e-10)
    flux = np.sqrt(np.sum(np.diff(S_norm, axis=1) ** 2, axis=0))

    def band_slope(mask):
        x = freqs[mask]
        y_ = 20 * np.log10(S[mask] + 1e-10)
        x_c = x - x.mean()
        denom = np.sum(x_c ** 2)
        y_c = y_ - y_.mean(axis=0, keepdims=True)
        return (x_c[:, None] * y_c).sum(axis=0) / denom

    slope_0_500    = band_slope((freqs >= 0) & (freqs <= 500))
    slope_500_1500 = band_slope((freqs > 500) & (freqs <= 1500))

    out = {}
    for name, arr in [('alpha_ratio', alpha_ratio), ('hammarberg_index', hammarberg),
                       ('spectral_flux', flux), ('spectral_slope_0_500', slope_0_500),
                       ('spectral_slope_500_1500', slope_500_1500)]:
        arr = np.asarray(arr, dtype=float)
        arr = arr[~np.isnan(arr)]
        out[f'{name}_mean'] = float(np.mean(arr)) if len(arr) else np.nan
        out[f'{name}_std']  = float(np.std(arr))  if len(arr) else np.nan
    return out


# ── Combine everything for ONE segment ────────────────────

def extract_all_segment(y_seg, sr=16000):
    snd = parselmouth.Sound(y_seg, sampling_frequency=sr)
    pitch     = snd.to_pitch_ac(pitch_floor=75, pitch_ceiling=600)
    intensity = snd.to_intensity(minimum_pitch=75)

    row = {}
    row.update(f0_features(pitch))
    row.update(intensity_features(intensity))
    row.update(energy_features(y_seg))
    row.update(pause_features(y_seg))
    row.update(articulation_features(snd, pitch, intensity))
    row.update(formant_features(snd))
    row.update(voice_quality_features(snd))
    row.update(mfcc_features(y_seg))
    row.update(spectral_shape_features(y_seg))
    return row


def extract_file_segments(filepath, uid, group_label, sr=16000):
    """Load one file, split into overlapping segments, extract features
    from each. Returns a list of row-dicts (one per segment)."""
    y, _ = librosa.load(filepath, sr=sr, mono=True)
    total_dur = len(y) / sr

    starts = np.arange(0, max(total_dur - MIN_SEGMENT_LEN, 0) + 1e-9, HOP_LEN)
    rows = []
    for seg_idx, start in enumerate(starts):
        end = min(start + SEGMENT_LEN, total_dur)
        if end - start < MIN_SEGMENT_LEN:
            continue
        y_seg = y[int(start * sr):int(end * sr)]
        try:
            feat = extract_all_segment(y_seg, sr)
            feat['uid'] = uid
            feat['segment_idx'] = seg_idx
            feat['group'] = group_label
            rows.append(feat)
        except Exception as e:
            print(f"    ERROR ({uid} seg{seg_idx}): {e}")
    return rows


# ── Metadata-driven file discovery (same as full-recording script) ──

def build_index(folder):
    exact, prefix_list = {}, []
    for root, _, files in os.walk(folder):
        for f in files:
            if f.endswith(AUDIO_EXTS):
                path = os.path.join(root, f)
                exact[Path(f).stem] = path
                prefix_list.append((f, path))
    return exact, prefix_list


def find_audio(uid, exact, prefix_list):
    if uid in exact:
        return exact[uid]
    for fname, path in prefix_list:
        if fname.startswith(uid):
            return path
    return None


def process_group(df_rows, exact, prefix_list, split, group_label, n_jobs=-1):
    out_dir = os.path.join(OUTPUT_DIR, split, group_label.lower())
    os.makedirs(out_dir, exist_ok=True)

    tasks = []
    for _, r in df_rows.iterrows():
        uid = r['uid']
        path = find_audio(uid, exact, prefix_list)
        if not path:
            print(f"  [!] {split}/{group_label} — not found: {uid}")
            continue
        tasks.append((uid, path))

    print(f"  Segmenting {len(tasks)} files in parallel...")
    results = Parallel(n_jobs=n_jobs, backend='threading', verbose=5)(
        delayed(extract_file_segments)(path, uid, group_label) for uid, path in tasks
    )
    rows = [row for file_rows in results for row in file_rows]  # flatten

    df_out = pd.DataFrame(rows)
    df_out.to_csv(os.path.join(out_dir, 'features_segments.csv'), index=False)
    n_recordings = df_out['uid'].nunique() if len(df_out) else 0
    print(f"  Saved {len(df_out)} segments from {n_recordings} recordings → {out_dir}/features_segments.csv\n")
    return df_out


# ── Main ───────────────────────────────────────────────────

meta = pd.read_csv(METADATA)
meta_eng = meta[meta['language'] == 'english']

for split, folder in [('train', TRAIN_FOLDER), ('test', TEST_FOLDER)]:
    print(f"\n{'='*50}\n{split.upper()}\n{'='*50}")
    split_df = meta_eng[meta_eng['split'] == split]

    cn_df = split_df[split_df['diagnosis'] == 'Control']
    ad_df = split_df[split_df['diagnosis'].isin(['ProbableAD', 'AD'])]

    print(f"Pool — CN: {len(cn_df)}  AD: {len(ad_df)}")
    print(f"Indexing {folder} ...")
    exact, prefix_list = build_index(folder)
    print(f"  {len(prefix_list)} audio files found")

    process_group(cn_df, exact, prefix_list, split, 'CN')
    process_group(ad_df, exact, prefix_list, split, 'AD')

print("\nDone.")


TRAIN
Pool — CN: 760  AD: 423
Indexing C:\Users\arday\Downloads\train_audios ...
  1646 audio files found
  Segmenting 760 files in parallel...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    7.3s
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 8.854874499982778 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:  1.1min
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 6.062383936721396 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 7.076399928417345 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_2694

  Saved 5349 segments from 760 recordings → C:\Users\arday\Desktop\full_features_segments\train\cn/features_segments.csv

  Segmenting 423 files in parallel...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    4.3s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   50.0s
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 9.325949270908275 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
[Parallel(n_jobs=-1)]: Done 146 tasks      | elapsed:  2.1min
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 9.282549680257368 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 7.480308235826847 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silenc

  Saved 2883 segments from 422 recordings → C:\Users\arday\Desktop\full_features_segments\train\ad/features_segments.csv


TEST
Pool — CN: 184  AD: 114
Indexing C:\Users\arday\Downloads\test_audios ...
  412 audio files found
  Segmenting 184 files in parallel...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    8.6s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   57.9s
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 9.54341475492565 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 7.716855018791875 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\1608240175.py:151: PraatWarning: The loudest and softest part in your sound differ by only 8.705539944864242 dB.
  tg = call(intensity, "To TextGrid (silences)", -35, 0.25, 0.1, "silence", "speech")
C:\Users\arday\AppData\Local\Temp\ipykernel_26948

  Saved 1310 segments from 184 recordings → C:\Users\arday\Desktop\full_features_segments\test\cn/features_segments.csv

  Segmenting 114 files in parallel...


[Parallel(n_jobs=-1)]: Done   2 tasks      | elapsed:    4.2s
[Parallel(n_jobs=-1)]: Done  56 tasks      | elapsed:   53.0s


  Saved 746 segments from 114 recordings → C:\Users\arday\Desktop\full_features_segments\test\ad/features_segments.csv


Done.


[Parallel(n_jobs=-1)]: Done 114 out of 114 | elapsed:  1.6min finished


In [3]:
"""
Step 1 (segments) — Merge, Clean, and Scale Segment-Level Feature Data
---------------------------------------------------------------------------
pip install pandas numpy scikit-learn joblib

Same as the recording-level Step 1, but keeps 'uid' and 'segment_idx'
so segments can later be grouped by person (GroupKFold) and predictions
can be aggregated back to person-level after training.
"""

import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler
import joblib

# ── Paths ───────────────────────────────────────────────
TRAIN_CN_CSV = r"C:\Users\arday\Desktop\full_features_segments\train\cn\features_segments.csv"
TRAIN_AD_CSV = r"C:\Users\arday\Desktop\full_features_segments\train\ad\features_segments.csv"
TEST_CN_CSV  = r"C:\Users\arday\Desktop\full_features_segments\test\cn\features_segments.csv"
TEST_AD_CSV  = r"C:\Users\arday\Desktop\full_features_segments\test\ad\features_segments.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features_segments\processed"
# ────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)

train_cn = pd.read_csv(TRAIN_CN_CSV); train_cn['label'] = 0
train_ad = pd.read_csv(TRAIN_AD_CSV); train_ad['label'] = 1
test_cn  = pd.read_csv(TEST_CN_CSV);  test_cn['label']  = 0
test_ad  = pd.read_csv(TEST_AD_CSV);  test_ad['label']  = 1

train_df = pd.concat([train_cn, train_ad], ignore_index=True)
test_df  = pd.concat([test_cn, test_ad], ignore_index=True)

print(f"Train: {len(train_df)} segments from {train_df['uid'].nunique()} recordings "
      f"(CN={train_df['uid'][train_df['label']==0].nunique()}, "
      f"AD={train_df['uid'][train_df['label']==1].nunique()})")
print(f"Test:  {len(test_df)} segments from {test_df['uid'].nunique()} recordings "
      f"(CN={test_df['uid'][test_df['label']==0].nunique()}, "
      f"AD={test_df['uid'][test_df['label']==1].nunique()})")

meta_cols = ['uid', 'segment_idx', 'group', 'label']
feature_cols = [c for c in train_df.columns if c not in meta_cols]

missing_frac = train_df[feature_cols].isna().mean()
drop_cols = missing_frac[missing_frac > 0.3].index.tolist()
if drop_cols:
    print(f"\nDropping {len(drop_cols)} features with >30% missing in train:")
    print(f"  {drop_cols}")
feature_cols = [c for c in feature_cols if c not in drop_cols]

train_median = train_df[feature_cols].median()
train_df[feature_cols] = train_df[feature_cols].fillna(train_median)
test_df[feature_cols]  = test_df[feature_cols].fillna(train_median)

scaler = RobustScaler()
train_scaled = scaler.fit_transform(train_df[feature_cols])
test_scaled  = scaler.transform(test_df[feature_cols])

train_out = pd.DataFrame(train_scaled, columns=feature_cols)
train_out['uid']         = train_df['uid'].values
train_out['segment_idx'] = train_df['segment_idx'].values
train_out['label']       = train_df['label'].values

test_out = pd.DataFrame(test_scaled, columns=feature_cols)
test_out['uid']         = test_df['uid'].values
test_out['segment_idx'] = test_df['segment_idx'].values
test_out['label']       = test_df['label'].values

train_out.to_csv(os.path.join(OUTPUT_DIR, 'train_segments_processed.csv'), index=False)
test_out.to_csv(os.path.join(OUTPUT_DIR, 'test_segments_processed.csv'), index=False)
joblib.dump(scaler, os.path.join(OUTPUT_DIR, 'scaler_segments.pkl'))
joblib.dump(feature_cols, os.path.join(OUTPUT_DIR, 'feature_cols_segments.pkl'))

print(f"\nFinal feature count: {len(feature_cols)}")
print(f"Saved → {OUTPUT_DIR}\\train_segments_processed.csv")
print(f"Saved → {OUTPUT_DIR}\\test_segments_processed.csv")
print(f"Saved → {OUTPUT_DIR}\\scaler_segments.pkl")
print(f"Saved → {OUTPUT_DIR}\\feature_cols_segments.pkl")

C:\Users\arday\AppData\Local\Temp\ipykernel_26948\2633861471.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_cn = pd.read_csv(TRAIN_CN_CSV); train_cn['label'] = 0
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\2633861471.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_ad = pd.read_csv(TRAIN_AD_CSV); train_ad['label'] = 1
C:\Users\arday\AppData\Local\Temp\ipykernel_26948\2633861471.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, 

Train: 8232 segments from 1182 recordings (CN=760, AD=422)
Test:  2056 segments from 298 recordings (CN=184, AD=114)

Final feature count: 135
Saved → C:\Users\arday\Desktop\full_features_segments\processed\train_segments_processed.csv
Saved → C:\Users\arday\Desktop\full_features_segments\processed\test_segments_processed.csv
Saved → C:\Users\arday\Desktop\full_features_segments\processed\scaler_segments.pkl
Saved → C:\Users\arday\Desktop\full_features_segments\processed\feature_cols_segments.pkl


In [4]:
"""
Step 2 (segments) — Statistical Analysis on Segment-Level Train Data
------------------------------------------------------------------------
pip install pandas numpy scipy

Same Cohen's d / Welch t-test approach as the recording-level analysis,
but now each segment counts as one observation. Note: since segments
from the same person are correlated, these p-values are optimistic
(pseudo-replication) — treat d as the primary signal, same as before,
and don't over-interpret p here. This step is mainly to see whether
segment-level rankings differ from the recording-level ones.
"""

import os
import pandas as pd
import numpy as np
from scipy import stats

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV  = r"C:\Users\arday\Desktop\full_features_segments\processed\train_segments_processed.csv"
OUTPUT_DIR = r"C:\Users\arday\Desktop\full_features_segments\processed"
# ────────────────────────────────────────────────────────

df = pd.read_csv(TRAIN_CSV)
feature_cols = [c for c in df.columns if c not in ('uid', 'segment_idx', 'label')]

cn = df[df['label'] == 0]
ad = df[df['label'] == 1]
print(f"Segments — CN: {len(cn)}   AD: {len(ad)}   Features: {len(feature_cols)}\n")

results = []
for f in feature_cols:
    c = cn[f].dropna().values
    a = ad[f].dropna().values
    if len(c) < 2 or len(a) < 2:
        continue

    pooled = np.sqrt((np.var(c) + np.var(a)) / 2)
    d = (np.mean(a) - np.mean(c)) / pooled if pooled > 0 else 0.0
    _, p = stats.ttest_ind(c, a, equal_var=False)

    results.append({
        'feature': f, 'd': d, 'abs_d': abs(d), 'p': p, 'significant': p < 0.05,
        'mean_cn': np.mean(c), 'mean_ad': np.mean(a),
    })

res_df = pd.DataFrame(results).sort_values('abs_d', ascending=False).reset_index(drop=True)
out_path = os.path.join(OUTPUT_DIR, 'feature_stats_segments.csv')
res_df.to_csv(out_path, index=False)

print(f"{'feature':<32}{'d':>8}{'p':>10}{'sig':>6}")
print("=" * 58)
for _, r in res_df.head(30).iterrows():
    flag = '*' if r['significant'] else ''
    print(f"{r['feature']:<32}{r['d']:>8.2f}{r['p']:>10.4f}{flag:>6}")

print(f"\nTotal features:          {len(res_df)}")
print(f"Large effect (|d|>=0.5):  {(res_df['abs_d']>=0.5).sum()}")
print(f"Medium effect (0.3-0.5):  {((res_df['abs_d']>=0.3)&(res_df['abs_d']<0.5)).sum()}")
print(f"\nSaved → {out_path}")

Segments — CN: 5349   AD: 2883   Features: 135

feature                                d         p   sig
spectral_slope_0_500_mean           0.38    0.0000     *
hammarberg_index_mean              -0.32    0.0000     *
mfcc_12_mean                       -0.31    0.0000     *
spectral_flux_mean                 -0.28    0.0000     *
spectral_flux_std                  -0.28    0.0000     *
mfcc_7_mean                        -0.27    0.0000     *
hammarberg_index_std                0.27    0.0000     *
mfcc_5_mean                        -0.26    0.0000     *
delta_mfcc_6_std                    0.26    0.0000     *
intensity_skewness                  0.25    0.0000     *
mfcc_13_mean                       -0.25    0.0000     *
mfcc_6_std                          0.24    0.0000     *
mfcc_11_mean                       -0.24    0.0000     *
mfcc_9_std                          0.24    0.0000     *
mfcc_7_std                          0.24    0.0000     *
intensity_std                       0.24

In [5]:
"""
Step 3 (segments) — Feature Selection: Effect Size Filtering + Correlation Pruning
---------------------------------------------------------------------------------------
pip install pandas numpy

Same greedy pruning approach as the recording-level Step 3: start from
features with |d| >= D_THRESHOLD, then walk from strongest to weakest,
dropping any feature too correlated with one already kept.
"""

import os
import pandas as pd
import numpy as np

# ── Paths & thresholds ───────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features_segments\processed\train_segments_processed.csv"
STATS_CSV    = r"C:\Users\arday\Desktop\full_features_segments\processed\feature_stats_segments.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features_segments\processed"
D_THRESHOLD    = 0.2
CORR_THRESHOLD = 0.85
# ─────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
stats_df = pd.read_csv(STATS_CSV)

candidates = stats_df[stats_df['abs_d'] >= D_THRESHOLD].sort_values('abs_d', ascending=False)
cand_features = candidates['feature'].tolist()
print(f"Candidates with |d| >= {D_THRESHOLD}: {len(cand_features)}")

corr = train_df[cand_features].corr().abs()

d_lookup = dict(zip(candidates['feature'], candidates['abs_d']))
kept, dropped = [], []

for f in cand_features:
    if f in dropped:
        continue
    too_correlated_with_kept = any(corr.loc[f, k] > CORR_THRESHOLD for k in kept)
    if too_correlated_with_kept:
        dropped.append(f)
        continue
    kept.append(f)
    for other in cand_features:
        if other != f and other not in kept and other not in dropped:
            if corr.loc[f, other] > CORR_THRESHOLD:
                dropped.append(other)

print(f"\nFinal feature count after correlation pruning: {len(kept)}")
print(f"Dropped as redundant: {len(dropped)}\n")

print(f"{'feature':<32}{'d':>8}")
print("=" * 40)
for f in kept:
    print(f"{f:<32}{d_lookup[f]:>8.3f}")

pd.Series(kept).to_csv(os.path.join(OUTPUT_DIR, 'selected_features_segments.csv'), index=False, header=['feature'])
print(f"\nSaved → {OUTPUT_DIR}\\selected_features_segments.csv")

Candidates with |d| >= 0.2: 34

Final feature count after correlation pruning: 25
Dropped as redundant: 9

feature                                d
spectral_slope_0_500_mean          0.381
hammarberg_index_mean              0.318
mfcc_12_mean                       0.315
spectral_flux_mean                 0.281
mfcc_7_mean                        0.273
hammarberg_index_std               0.266
mfcc_5_mean                        0.264
delta_mfcc_6_std                   0.258
intensity_skewness                 0.252
mfcc_13_mean                       0.251
mfcc_11_mean                       0.244
mfcc_9_std                         0.243
mfcc_7_std                         0.243
intensity_std                      0.242
mfcc_11_std                        0.241
energy_skewness                    0.238
hnr_min                            0.237
delta_mfcc_8_std                   0.234
mfcc_12_std                        0.228
mfcc_9_mean                        0.227
mfcc_10_std                     

In [6]:
"""
Step 4 (segments) — Baseline Models with Grouped Cross-Validation
------------------------------------------------------------------------
pip install pandas numpy scikit-learn

Uses StratifiedGroupKFold (groups=uid) instead of plain StratifiedKFold —
this keeps every segment from the same person in the same fold, so the
model is never validated on a person it partially trained on.

Segment-level out-of-fold probabilities are then averaged PER PERSON to
get one final probability per recording, since the actual diagnostic
decision is made per person, not per 5-second clip. All metrics are
reported at this aggregated, person level so they're directly
comparable to the recording-level results from Step 4.
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features_segments\processed\train_segments_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features_segments\processed\selected_features_segments.csv"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

df = pd.read_csv(TRAIN_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X = df[features].values
y = df['label'].values
groups = df['uid'].values

n_people = df['uid'].nunique()
print(f"Segments: {len(y)}   People: {n_people}   Features: {len(features)}")
print(f"Segment labels — CN={sum(y==0)}, AD={sum(y==1)}\n")

models = {
    'Logistic Regression': LogisticRegression(class_weight='balanced', max_iter=1000, random_state=RANDOM_STATE),
    'SVM (linear)':        SVC(kernel='linear', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'SVM (rbf)':           SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE),
}


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

results = []
for name, model in models.items():
    print(f"Running {name} ...")
    oof_proba = np.zeros(len(y))

    for train_idx, val_idx in sgkf.split(X, y, groups=groups):
        model.fit(X[train_idx], y[train_idx])
        oof_proba[val_idx] = model.predict_proba(X[val_idx])[:, 1]

    # segment-level AUC, just for reference
    segment_auc = roc_auc_score(y, oof_proba)

    # aggregate to PERSON level: mean probability per uid
    agg = pd.DataFrame({'uid': groups, 'y': y, 'proba': oof_proba})
    person = agg.groupby('uid').agg(y=('y', 'first'), proba=('proba', 'mean')).reset_index()

    person_auc = roc_auc_score(person['y'], person['proba'])
    fpr, tpr, thresholds = roc_curve(person['y'], person['proba'])
    best_thresh = thresholds[np.argmax(tpr - fpr)]
    sens, spec, bal = metrics_at_threshold(person['y'], person['proba'], best_thresh)

    results.append({
        'model': name, 'segment_AUC': segment_auc, 'person_AUC': person_auc,
        'best_thresh': best_thresh, 'sens_opt': sens, 'spec_opt': spec, 'bal_opt': bal,
    })
    print(f"  segment AUC={segment_auc:.3f}   person AUC={person_auc:.3f}   person BalAcc={bal:.3f}\n")

res_df = pd.DataFrame(results).sort_values('bal_opt', ascending=False)

print(f"\n{'Model':<22}{'SegAUC':>8}{'PersonAUC':>11}{'Thr':>7}{'SensOpt':>9}{'SpecOpt':>9}{'BalOpt':>9}")
print("=" * 78)
for _, r in res_df.iterrows():
    print(f"{r['model']:<22}{r['segment_AUC']:>8.3f}{r['person_AUC']:>11.3f}{r['best_thresh']:>7.3f}"
          f"{r['sens_opt']:>9.3f}{r['spec_opt']:>9.3f}{r['bal_opt']:>9.3f}")

Segments: 8232   People: 1182   Features: 25
Segment labels — CN=5349, AD=2883

Running Logistic Regression ...
  segment AUC=0.641   person AUC=0.647   person BalAcc=0.635

Running SVM (linear) ...
  segment AUC=0.643   person AUC=0.643   person BalAcc=0.638

Running SVM (rbf) ...
  segment AUC=0.651   person AUC=0.658   person BalAcc=0.644

Running Random Forest ...
  segment AUC=0.666   person AUC=0.681   person BalAcc=0.644


Model                   SegAUC  PersonAUC    Thr  SensOpt  SpecOpt   BalOpt
SVM (rbf)                0.651      0.658  0.398    0.566    0.722    0.644
Random Forest            0.666      0.681  0.338    0.637    0.650    0.644
SVM (linear)             0.643      0.643  0.377    0.571    0.704    0.638
Logistic Regression      0.641      0.647  0.476    0.680    0.591    0.635


In [7]:
"""
Step 5 (segments) — Final Model: Train on Full Train Segments,
Evaluate Once on Test Set (Person-Level)
------------------------------------------------------------------------
pip install pandas numpy scikit-learn joblib

Trains Random Forest — the best-performing model from the segment-level
Step 4 — on ALL train segments. The classification threshold is chosen
using grouped out-of-fold predictions on the train set only (aggregated
to person-level), then applied as a FIXED threshold on the test set.
Test segment probabilities are averaged per person before computing
final metrics, since the diagnosis is made per person, not per segment.
"""

import os
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, confusion_matrix, balanced_accuracy_score, roc_curve

# ── Paths ───────────────────────────────────────────────
TRAIN_CSV    = r"C:\Users\arday\Desktop\full_features_segments\processed\train_segments_processed.csv"
TEST_CSV     = r"C:\Users\arday\Desktop\full_features_segments\processed\test_segments_processed.csv"
FEATURES_CSV = r"C:\Users\arday\Desktop\full_features_segments\processed\selected_features_segments.csv"
OUTPUT_DIR   = r"C:\Users\arday\Desktop\full_features_segments\processed"
N_FOLDS      = 5
RANDOM_STATE = 42
# ────────────────────────────────────────────────────────

train_df = pd.read_csv(TRAIN_CSV)
test_df  = pd.read_csv(TEST_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()

X_train, y_train, groups_train = train_df[features].values, train_df['label'].values, train_df['uid'].values
X_test,  y_test,  groups_test  = test_df[features].values,  test_df['label'].values,  test_df['uid'].values

print(f"Train: {len(y_train)} segments from {train_df['uid'].nunique()} people")
print(f"Test:  {len(y_test)} segments from {test_df['uid'].nunique()} people\n")


def metrics_at_threshold(y_true, proba, thresh):
    pred = (proba >= thresh).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    spec = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    bal  = balanced_accuracy_score(y_true, pred)
    return sens, spec, bal


def to_person_level(uids, y, proba):
    agg = pd.DataFrame({'uid': uids, 'y': y, 'proba': proba})
    return agg.groupby('uid').agg(y=('y', 'first'), proba=('proba', 'mean')).reset_index()


# ── Step A: find the threshold using TRAIN-only grouped out-of-fold CV ──
print("Finding threshold via grouped 5-fold CV on train segments...")
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
oof_proba = np.zeros(len(y_train))

for train_idx, val_idx in sgkf.split(X_train, y_train, groups=groups_train):
    model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE)
    model.fit(X_train[train_idx], y_train[train_idx])
    oof_proba[val_idx] = model.predict_proba(X_train[val_idx])[:, 1]

train_person = to_person_level(groups_train, y_train, oof_proba)
train_person_auc = roc_auc_score(train_person['y'], train_person['proba'])

fpr, tpr, thresholds = roc_curve(train_person['y'], train_person['proba'])
threshold = thresholds[np.argmax(tpr - fpr)]
print(f"  Train CV person-level AUC: {train_person_auc:.3f}")
print(f"  Chosen threshold:          {threshold:.3f}  (from train CV only, test untouched)\n")

# ── Step B: train the FINAL model on ALL train segments ──
print("Training final model on full train segments...")
final_model = RandomForestClassifier(n_estimators=300, class_weight='balanced', random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)

# ── Step C: evaluate ONCE on the test set, aggregated to person level ──
test_seg_proba = final_model.predict_proba(X_test)[:, 1]
test_person = to_person_level(groups_test, y_test, test_seg_proba)

test_auc = roc_auc_score(test_person['y'], test_person['proba'])
sens, spec, bal = metrics_at_threshold(test_person['y'], test_person['proba'], threshold)

print(f"\n{'='*55}")
print("FINAL TEST SET RESULTS — person level (evaluated once)")
print(f"{'='*55}")
print(f"  People in test set: {len(test_person)}")
print(f"  AUC:                {test_auc:.3f}")
print(f"  Threshold used:     {threshold:.3f}")
print(f"  Sensitivity:        {sens:.3f}")
print(f"  Specificity:        {spec:.3f}")
print(f"  Balanced Accuracy:  {bal:.3f}")

joblib.dump(final_model, os.path.join(OUTPUT_DIR, 'final_model_rf_segments.pkl'))
joblib.dump(threshold, os.path.join(OUTPUT_DIR, 'final_threshold_segments.pkl'))
print(f"\nSaved → {OUTPUT_DIR}\\final_model_rf_segments.pkl")
print(f"Saved → {OUTPUT_DIR}\\final_threshold_segments.pkl")

Train: 8232 segments from 1182 people
Test:  2056 segments from 298 people

Finding threshold via grouped 5-fold CV on train segments...
  Train CV person-level AUC: 0.681
  Chosen threshold:          0.338  (from train CV only, test untouched)

Training final model on full train segments...

FINAL TEST SET RESULTS — person level (evaluated once)
  People in test set: 298
  AUC:                0.682
  Threshold used:     0.338
  Sensitivity:        0.614
  Specificity:        0.668
  Balanced Accuracy:  0.641

Saved → C:\Users\arday\Desktop\full_features_segments\processed\final_model_rf_segments.pkl
Saved → C:\Users\arday\Desktop\full_features_segments\processed\final_threshold_segments.pkl


In [8]:
"""
Step 7 (segments) — Interpretability: SHAP + Person-Level Permutation Importance
------------------------------------------------------------------------------------
pip install pandas numpy scikit-learn shap matplotlib joblib

Random Forest lets us use shap.TreeExplainer — exact and fast, no need
for the sampling-based KernelExplainer we had to use for SVM (rbf).

Permutation importance is computed with a CUSTOM loop rather than
sklearn's built-in function: our real evaluation metric is person-level
AUC (segment probabilities averaged per uid), so a feature's importance
should be measured by how much PERSON-level AUC drops when it's
shuffled — not segment-level AUC, which could give a misleading ranking.
"""

import os
import numpy as np
import pandas as pd
import joblib
import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

# ── Paths ───────────────────────────────────────────────
TEST_CSV       = r"C:\Users\arday\Desktop\full_features_segments\processed\test_segments_processed.csv"
FEATURES_CSV   = r"C:\Users\arday\Desktop\full_features_segments\processed\selected_features_segments.csv"
MODEL_PATH     = r"C:\Users\arday\Desktop\full_features_segments\processed\final_model_rf_segments.pkl"
OUTPUT_DIR     = r"C:\Users\arday\Desktop\full_features_segments\processed"
N_REPEATS      = 15
RANDOM_STATE   = 42
# ────────────────────────────────────────────────────────

test_df  = pd.read_csv(TEST_CSV)
features = pd.read_csv(FEATURES_CSV)['feature'].tolist()
model    = joblib.load(MODEL_PATH)

X_test = test_df[features].values
y_test = test_df['label'].values
uids   = test_df['uid'].values


def to_person_level(uids, y, proba):
    agg = pd.DataFrame({'uid': uids, 'y': y, 'proba': proba})
    return agg.groupby('uid').agg(y=('y', 'first'), proba=('proba', 'mean')).reset_index()


def person_auc(X):
    proba = model.predict_proba(X)[:, 1]
    person = to_person_level(uids, y_test, proba)
    return roc_auc_score(person['y'], person['proba'])


# ── 1. Person-level permutation importance ──
print("Computing person-level permutation importance...")
rng = np.random.RandomState(RANDOM_STATE)
baseline_auc = person_auc(X_test)
print(f"  Baseline person-level AUC: {baseline_auc:.3f}")

perm_results = []
for i, feat in enumerate(features):
    drops = []
    for _ in range(N_REPEATS):
        X_perm = X_test.copy()
        X_perm[:, i] = X_perm[rng.permutation(len(X_perm)), i]
        drops.append(baseline_auc - person_auc(X_perm))
    perm_results.append({'feature': feat, 'perm_importance_mean': np.mean(drops),
                          'perm_importance_std': np.std(drops)})
    print(f"  [{i+1}/{len(features)}] {feat}: {np.mean(drops):.4f}")

perm_df = pd.DataFrame(perm_results).sort_values('perm_importance_mean', ascending=False)

# ── 2. SHAP (TreeExplainer — exact, no sampling needed for Random Forest) ──
print("\nComputing SHAP values...")
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

if isinstance(shap_values, list):
    sv = shap_values[1]            # class 1 = AD (older shap versions)
elif shap_values.ndim == 3:
    sv = shap_values[:, :, 1]      # (n_samples, n_features, n_classes) — newer shap versions
else:
    sv = shap_values

shap_df = pd.DataFrame({
    'feature': features,
    'mean_abs_shap': np.abs(sv).mean(axis=0),
}).sort_values('mean_abs_shap', ascending=False)

# ── Combine and print ──
combined = perm_df.merge(shap_df, on='feature').sort_values('mean_abs_shap', ascending=False)

print(f"\n{'feature':<28}{'SHAP':>10}{'PermImp':>10}")
print("=" * 48)
for _, r in combined.iterrows():
    print(f"{r['feature']:<28}{r['mean_abs_shap']:>10.4f}{r['perm_importance_mean']:>10.4f}")

combined.to_csv(os.path.join(OUTPUT_DIR, 'feature_importance_segments.csv'), index=False)


# ── Plots ──
def dark_barplot(df, value_col, title, filename, top_n=25):
    top = df.head(top_n).iloc[::-1]
    fig, ax = plt.subplots(figsize=(8, 7))
    fig.patch.set_facecolor('#0d1117'); ax.set_facecolor('#111827')
    ax.barh(top['feature'], top[value_col], color='#60a5fa')
    ax.set_title(title, color='white', fontsize=12)
    ax.tick_params(colors='#94a3b8', labelsize=8)
    for spine in ax.spines.values():
        spine.set_color('#1e293b')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename), dpi=150, facecolor='#0d1117')
    plt.close()


dark_barplot(shap_df, 'mean_abs_shap', 'Top Features — Mean |SHAP value| (segments)', 'shap_importance_segments.png')
dark_barplot(perm_df, 'perm_importance_mean', 'Top Features — Person-Level Permutation Importance', 'permutation_importance_segments.png')

print(f"\nSaved → {OUTPUT_DIR}\\feature_importance_segments.csv")
print(f"Saved → {OUTPUT_DIR}\\shap_importance_segments.png")
print(f"Saved → {OUTPUT_DIR}\\permutation_importance_segments.png")

C:\Users\arday\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Computing person-level permutation importance...
  Baseline person-level AUC: 0.682
  [1/25] spectral_slope_0_500_mean: 0.0026
  [2/25] hammarberg_index_mean: 0.0027
  [3/25] mfcc_12_mean: 0.0009
  [4/25] spectral_flux_mean: 0.0082
  [5/25] mfcc_7_mean: 0.0033
  [6/25] hammarberg_index_std: 0.0031
  [7/25] mfcc_5_mean: -0.0024
  [8/25] delta_mfcc_6_std: -0.0018
  [9/25] intensity_skewness: 0.0105
  [10/25] mfcc_13_mean: 0.0075
  [11/25] mfcc_11_mean: 0.0010
  [12/25] mfcc_9_std: 0.0022
  [13/25] mfcc_7_std: 0.0012
  [14/25] intensity_std: 0.0085
  [15/25] mfcc_11_std: 0.0015
  [16/25] energy_skewness: 0.0078
  [17/25] hnr_min: 0.0025
  [18/25] delta_mfcc_8_std: -0.0037
  [19/25] mfcc_12_std: -0.0003
  [20/25] mfcc_9_mean: 0.0006
  [21/25] mfcc_10_std: -0.0032
  [22/25] intensity_kurtosis: 0.0064
  [23/25] intensity_mean: 0.0108
  [24/25] delta_mfcc_13_std: -0.0018
  [25/25] f0_std: 0.0060

Computing SHAP values...

feature                           SHAP   PermImp
spectral_slope_0_500_m